In [2]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

In [3]:
data = pd.read_csv("cleaned_heart.csv")

print("Shape:", data.shape)
data.head()

Shape: (920, 20)


,id,age,trestbps,chol,fbs,thalch,exang,oldpeak,num,sex_Male,dataset_Hungary,dataset_Switzerland,dataset_VA Long Beach,cp_atypical angina,cp_non-anginal,cp_typical angina,restecg_normal,restecg_st-t abnormality,slope_flat,slope_upsloping
0,1,63,145.0,233.0,True,150.0,False,2.3,0,True,False,False,False,False,False,True,False,False,False,False
1,2,67,160.0,286.0,False,108.0,True,1.5,2,True,False,False,False,False,False,False,False,False,True,False
2,3,67,120.0,229.0,False,129.0,True,2.6,1,True,False,False,False,False,False,False,False,False,True,False
3,4,37,130.0,250.0,False,187.0,False,3.5,0,True,False,False,False,False,True,False,True,False,False,False
4,5,41,130.0,204.0,False,172.0,False,1.4,0,False,False,False,False,True,False,False,False,False,False,True


In [4]:
data['risk'] = data['num'].apply(lambda x: 1 if x > 0 else 0)

data['risk'].value_counts()

risk
1    509
0    411
Name: count, dtype: int64

In [5]:
drop_cols = ['num', 'risk']

if 'id' in data.columns:
    drop_cols.append('id')

X = data.drop(columns=drop_cols)
y = data['risk']

print("Features:", X.columns.tolist())

Features: ['age', 'trestbps', 'chol', 'fbs', 'thalch', 'exang', 'oldpeak', 'sex_Male', 'dataset_Hungary', 'dataset_Switzerland', 'dataset_VA Long Beach', 'cp_atypical angina', 'cp_non-anginal', 'cp_typical angina', 'restecg_normal', 'restecg_st-t abnormality', 'slope_flat', 'slope_upsloping']


In [6]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler 🔥
joblib.dump(scaler, "scaler.pkl")

print("✅ Scaler saved")

✅ Scaler saved


In [7]:
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [8]:
def generate_sequences(data, labels, timesteps=15):
    X_seq = []
    y_seq = []

    for i in range(len(data)):
        row = data.iloc[i].values
        sequence = []

        for t in range(timesteps):
            step = row.copy()

            # progression
            step = step * (1 + t * 0.03)

            # noise
            step += np.random.normal(0, 0.01, size=row.shape)

            sequence.append(step)

        X_seq.append(sequence)
        y_seq.append(labels.iloc[i])

    return np.array(X_seq), np.array(y_seq)

X_lstm, y_lstm = generate_sequences(X_scaled, y)

print("Shape:", X_lstm.shape)

Shape: (920, 15, 18)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X_lstm, y_lstm,
    test_size=0.2,
    random_state=42
)

In [10]:
model = Sequential()

model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))
model.add(LSTM(64))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 64)                21248     
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense (Dense)               (None, 1)                 65        
                                                                 
Total params: 21313 (83.25 KB)
Trainable params: 21313 (83.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [11]:
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/15
19/19 [==============================] - 3s 41ms/step - loss: 0.6180 - accuracy: 0.6599 - val_loss: 0.4908 - val_accuracy: 0.8311
Epoch 2/15
19/19 [==============================] - 0s 12ms/step - loss: 0.4540 - accuracy: 0.7959 - val_loss: 0.4060 - val_accuracy: 0.8446
Epoch 3/15
19/19 [==============================] - 0s 11ms/step - loss: 0.4335 - accuracy: 0.7942 - val_loss: 0.3908 - val_accuracy: 0.8378
Epoch 4/15
19/19 [==============================] - 0s 12ms/step - loss: 0.4278 - accuracy: 0.8061 - val_loss: 0.3924 - val_accuracy: 0.8446
Epoch 5/15
19/19 [==============================] - 0s 11ms/step - loss: 0.4141 - accuracy: 0.8112 - val_loss: 0.3892 - val_accuracy: 0.8378
Epoch 6/15
19/19 [==============================] - 0s 12ms/step - loss: 0.4110 - accuracy: 0.8197 - val_loss: 0.3826 - val_accuracy: 0.8514
Epoch 7/15
19/19 [==============================] - 0s 12ms/step - loss: 0.4021 - accuracy: 0.8163 - val_loss: 0.3907 - val_accuracy: 0.8378
Epoch 8/15
19

In [12]:
loss, acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", acc)

6/6 [==============================] - 0s 5ms/step - loss: 0.4281 - accuracy: 0.7880
Test Accuracy: 0.7880434989929199


In [16]:
model = load_model("lstm_model_15.h5")
scaler = joblib.load("scaler.pkl")

print("✅ Loaded successfully")

✅ Loaded successfully


In [17]:
feature_names = X.columns.tolist()

In [14]:
model.save("lstm_model_15.h5")

print("✅ Model saved")

✅ Model saved


C:\Users\Amitesh\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [21]:
def make_sequence(row, timesteps=15):
    seq = []
    for t in range(timesteps):
        step = row.copy()
        step = step * (1 + t * 0.03)
        step += np.random.normal(0, 0.01, size=row.shape)
        seq.append(step)
    return np.array(seq)

# ----------------------------
# 3. Input ONE new data row (18 features)
# IMPORTANT: must already be scaled OR scale it below
# ----------------------------
row = np.array([
    0.70, 0.72, 0.39, 0.51, 0.61, 0.45, 0.38, 0.27, 0.19,
    0.14, 0.10, 0.08, 0.05, 0.03, 0.01, -0.01, -0.007, -0.008
])

# ----------------------------
# 4. (Optional but critical) Apply SAME scaler used in training
# ----------------------------
row = scaler.transform(row.reshape(1, -1)).flatten()

# ----------------------------
# 5. Generate full sequence (15 timesteps)
# ----------------------------
sequence = make_sequence(row, timesteps=15)

# ----------------------------
# 6. Reshape for LSTM → (1, 15, 18)
# ----------------------------
x_new = sequence.reshape(1, 15, 18)

# ----------------------------
# 7. Predict
# ----------------------------
y_pred = model.predict(x_new)

print("Prediction:", y_pred)

1/1 [==============================] - 0s 34ms/step
Prediction: [[0.78152436]]


C:\Users\Amitesh\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
